# 0. Mon setup colab

In [218]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = True  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "pandas>=2.2,<4",
        "requests>=2.31,<3",
        "plotly>=6,<7",
        "ipywidgets>=8.1,<9",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")


Installation finished. Restart the session if Colab requests it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [219]:
!git clone https://github.com/ammou35/LNH-data-science.git

Cloning into 'LNH-data-science'...
remote: Enumerating objects: 402, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 402 (delta 64), reused 64 (delta 52), pack-reused 311 (from 1)
Receiving objects: 100% (402/402), 1.03 MiB | 2.86 MiB/s, done.
Resolving deltas: 100% (202/202), done.


In [226]:
%cd /content/LNH-data-science/

/content/LNH-data-science


In [227]:
!git switch advanced_viz

Already on 'advanced_viz'
Your branch is up to date with 'origin/advanced_viz'.


In [228]:
!git pull

remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 89 (delta 56), reused 59 (delta 47), pack-reused 18 (from 1)
Unpacking objects: 100% (89/89), 472.21 KiB | 1.54 MiB/s, done.
From https://github.com/ammou35/LNH-data-science
 * [new branch]      blog-task2-ipywidgets -> origin/blog-task2-ipywidgets
   7e81d79..fea3cbd  llm-code-generation   -> origin/llm-code-generation
   7900c76..50163bd  main                  -> origin/main
 * [new branch]      part_5_blog           -> origin/part_5_blog
 * [new branch]      simple_viz_blog       -> origin/simple_viz_blog
Already up to date.


In [223]:
import os
os.environ['NHL_DATA_DIR'] = '/content/drive/MyDrive/ift6758/data/raw'

In [229]:
%cd data_science_project/

/content/LNH-data-science/data_science_project


In [230]:
!uv sync

Resolved 179 packages in 3ms
Checked 129 packages in 9ms


In [231]:
# Setup
import sys, ift6758
print(sys.executable)
print(ift6758.__file__)

import json
import os
from pathlib import Path
import pandas as pd

from ipywidgets import interact, Dropdown
import matplotlib.pyplot as plt
from IPython.display import display

# Racine du projet = le dossier data_science_project (ce notebook est dans son sous-dossier notebooks/).
# Sert à trouver le cache de données (data/raw) et l'image de la patinoire (figures/nhl_rink.png).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "data_science_project")
os.environ["NHL_DATA_DIR"] = str(ROOT / "ift6758/data")
print("Data directory:", os.environ["NHL_DATA_DIR"])

from ift6758.data import get_game
from ift6758.data.nhl import GAME_TYPE_CODE, get_data_dir

/usr/bin/python3
/content/LNH-data-science/data_science_project/ift6758/__init__.py
Data directory: /content/LNH-data-science/data_science_project/ift6758/data


In [232]:
# https://plotly.com/python-api-reference/generated/plotly.express.html#module-plotly.express

# Import required packages
import plotly.graph_objects as go
import pandas as pd
from ipywidgets import Dropdown
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [200]:
# load dataset
# hacky, needs fixing
# Ma premiere maniere de bin, avec encore le tuple (x,y) en tant que colonne (c'est un string quand c'est importe de CSV donc faut faire des incantations pour l'avoir en x et y clean...)
old_df = pd.read_csv("/content/drive/MyDrive/ift6758/data/vizualisations/excess_shots_bin20x10_bak.csv",
                 usecols=['season', 'team','bin20x10', 'shot_per_team-hr'],
                 dtype={'season': str, 'team': str, 'x':np.float64, 'y':np.float64, 'shot_per_team-hr':np.float64})
old_df = old_df.assign(x_y=lambda df: [x.strip('()').split(',') for x in df['bin20x10']])
old_df[['x','y']] = pd.DataFrame(old_df['x_y'].tolist(), index= old_df.index, dtype=np.float64)

In [233]:
# load dataset
# Nouvelle maniere, plus simple, la separation est faite en amont
# Mais p-e c'est encore mieux d'avoir les donnees importees directement en x,y levels comme dans l'exemple
new_df = pd.read_csv("/content/drive/MyDrive/ift6758/data/vizualisations/excess_shots_bin20x10.csv",
                 usecols=['season', 'team','x', 'y', 'shot_per_team-hr'],
                 dtype={'season': str, 'team': str, 'x':np.float64, 'y':np.float64, 'shot_per_team-hr':np.float64})

In [234]:
new_df.groupby(['season', 'team'])['shot_per_team-hr'].describe()

count      mean       std       min       25%       50%  \
season team                                                            
2016   ANA   175.0  0.002780  0.154126 -0.503152 -0.058328 -0.002048   
       ARI   172.0 -0.011789  0.154443 -0.668833 -0.071712  0.002125   
       BOS   166.0  0.017216  0.162612 -0.498102 -0.054821 -0.001924   
       BUF   168.0  0.004031  0.163037 -0.571272 -0.070601  0.002940   
       CAR   155.0  0.004467  0.156451 -0.550855 -0.070453  0.003926   
...            ...       ...       ...       ...       ...       ...   
2023   TOR   178.0  0.014927  0.176438 -0.665674 -0.060366 -0.003600   
       VAN   186.0 -0.014657  0.126217 -0.541842 -0.058929 -0.006889   
       VGK   184.0  0.003996  0.127462 -0.665674 -0.036411  0.000783   
       WPG   188.0  0.000291  0.117385 -0.641782 -0.042322  0.000657   
       WSH   181.0 -0.018396  0.118837 -0.635233 -0.053920 -0.006944   

                  75%       max  
season team                      
2016   ANA   0.031390  0.777329  
       ARI   0.034166  0.487573  
       BOS   0.051535  0.778500  
       BUF   0.050915  0.707086  
       CAR   0.062953  0.805795  
...               ...       ...  
2023   TOR   0.057022  0.930742  
       VAN   0.024812  0.795075  
       VGK   0.038662  0.632095  
       WPG   0.031757  0.545952  
       WSH   0.019884  0.449053  

[250 rows x 8 columns]

In [203]:
old_df.groupby(['season', 'team'])[['shot_per_team-hr']].describe()

shot_per_team-hr                                          \
                       count      mean       std       min       25%   
season team                                                            
2016   ANA             175.0  0.002843  0.073030 -0.191620 -0.029532   
       ARI             175.0 -0.011834  0.078920 -0.386253 -0.041039   
       BOS             169.0  0.016857  0.079279 -0.276092 -0.011622   
       BUF             176.0  0.002241  0.070162 -0.149749 -0.027351   
       CAR             157.0  0.004480  0.066091 -0.277062 -0.022973   
...                      ...       ...       ...       ...       ...   
2023   TOR             182.0  0.013995  0.073279 -0.125140 -0.020259   
       VAN             188.0 -0.014716  0.055171 -0.173083 -0.039196   
       VGK             186.0  0.003867  0.054719 -0.224972 -0.027639   
       WPG             188.0  0.000182  0.059510 -0.223268 -0.027019   
       WSH             182.0 -0.018460  0.048441 -0.225756 -0.043121   

                                           
                  50%       75%       max  
season team                                
2016   ANA   0.001369  0.020662  0.397437  
       ARI  -0.001852  0.018024  0.290099  
       BOS   0.006868  0.037836  0.339623  
       BUF   0.000829  0.024923  0.344890  
       CAR   0.001278  0.030150  0.284303  
...               ...       ...       ...  
2023   TOR  -0.000467  0.037186  0.397030  
       VAN  -0.008534  0.008158  0.346917  
       VGK  -0.000042  0.031220  0.265072  
       WPG   0.000164  0.021116  0.272660  
       WSH  -0.012496  0.003223  0.201254  

[250 rows x 8 columns]

In [236]:
# Test de l'utilisation du @ dans les queries pour utiliser dynamiquement le nom des variables
season = '2017'
team = "SJS"
new_df.query('season == @season and team == @team').pivot(index="y", columns="x", values="shot_per_team-hr")

x,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0
y,,,,,,,,,,,,,,,,,,,,
0.0,NaN,-0.014223,-0.050955,0.042780,0.012017,-0.006000,-0.010966,0.002182,-0.010902,-0.002246,NaN,-0.012747,0.012682,0.024025,0.003762,-0.034045,-0.008279,0.042812,-0.015161,NaN
1.0,0.012514,-0.002679,-0.044778,0.059025,0.024138,0.118105,0.083218,0.046567,-0.016637,0.013589,-0.007043,-0.025293,-0.005567,0.052992,-0.096166,-0.172052,-0.120825,-0.079801,-0.003818,NaN
2.0,NaN,-0.060813,0.093045,0.204163,0.247377,0.174940,0.187318,0.006778,NaN,NaN,NaN,-0.013485,NaN,0.000826,-0.207741,-0.114544,-0.181606,-0.148131,-0.062658,0.000168
3.0,0.006241,0.064696,0.026159,0.148003,0.091938,0.025004,0.123536,0.026873,0.015635,-0.002046,NaN,0.007717,-0.003522,-0.057893,-0.110428,-0.099422,-0.182480,-0.264199,-0.205359,0.005503
4.0,0.011776,0.531622,0.775975,0.401532,0.103883,0.160910,0.139909,0.053746,-0.010902,-0.012747,-0.013116,NaN,-0.011840,-0.098516,-0.128341,-0.203385,-0.214744,-0.409418,-0.398155,-0.001677
5.0,0.021170,0.660180,0.447224,0.216076,0.074459,0.106794,0.045235,0.004195,0.002182,0.070889,-0.002984,NaN,-0.006674,-0.082617,-0.138240,-0.123945,-0.191802,-0.485850,-0.225028,0.023015
6.0,0.050826,0.168129,0.135930,0.118177,0.036916,0.042716,0.209418,0.021170,-0.004629,0.022646,NaN,-0.004629,0.003289,-0.053265,-0.077587,-0.156754,-0.147690,-0.157115,-0.110525,0.005503
7.0,0.008824,0.088609,0.036483,0.258712,0.133443,0.083419,0.323993,0.037743,0.023921,-0.005736,NaN,NaN,NaN,-0.096599,-0.141184,-0.226761,-0.235184,-0.132031,-0.054372,0.019694
8.0,0.006073,0.052776,0.034799,0.037285,0.191104,0.110589,0.616164,-0.000770,NaN,NaN,-0.013854,-0.017006,0.005302,-0.066204,-0.024475,-0.109217,-0.155214,-0.079601,-0.066822,NaN


In [204]:
old_df.query('season == @season and team == @team').pivot(index="y", columns="x", values="shot_per_team-hr")

x,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0
y,,,,,,,,,,,,,,,,,,,,
0.0,NaN,-0.009795,-0.053706,0.047746,0.019028,0.022381,-0.042837,0.011206,NaN,-0.014223,NaN,-0.010164,0.027242,0.026809,0.013725,0.023087,0.007958,0.068242,-0.001709,NaN
1.0,0.004227,-0.045789,-0.023576,-0.032159,0.032793,0.039804,0.016758,0.057067,-0.002278,0.015065,-0.014961,-0.022341,-0.002615,0.208519,-0.112169,-0.083050,-0.076207,-0.029312,0.044826,0.005335
2.0,0.013621,-0.062490,0.032087,0.080539,0.033307,0.020351,0.110862,0.005671,NaN,NaN,NaN,-0.010902,-0.001508,0.213990,-0.156546,-0.039532,-0.020039,-0.086235,-0.037261,0.009193
3.0,-0.003522,-0.050642,-0.044962,0.063228,0.063958,-0.016260,0.099414,0.016004,0.027980,0.000168,-0.000570,0.005872,-0.003153,0.076199,-0.053329,-0.043230,-0.010212,-0.070929,-0.131349,0.017111
4.0,0.013621,0.283299,0.257284,0.136804,0.053963,0.053690,0.125349,0.045460,-0.011271,-0.011640,-0.000401,NaN,-0.023448,-0.006193,-0.050409,-0.137799,0.024218,0.010332,0.133619,0.014897
5.0,0.004227,0.210565,0.175221,0.060581,0.008503,0.041946,-0.056113,0.024828,-0.001508,0.059281,-0.005198,NaN,-0.007412,-0.059033,-0.044104,-0.073656,-0.010541,-0.114905,-0.058864,0.021539
6.0,0.039219,0.094120,-0.057340,-0.054091,-0.019276,-0.014383,0.075325,0.020801,-0.002784,0.009193,NaN,NaN,0.014159,-0.029143,-0.036323,-0.128774,-0.062915,-0.085994,0.004813,0.015266
7.0,-0.000201,0.063212,-0.025413,0.097144,0.058431,0.032224,0.110829,0.016742,0.021338,-0.008319,NaN,NaN,NaN,-0.020143,0.013404,-0.012691,-0.111559,-0.071073,-0.052695,-0.001677
8.0,NaN,0.004131,-0.015691,-0.007332,0.102102,0.126592,0.460637,-0.003722,NaN,-0.019389,-0.015699,NaN,-0.005198,0.000257,0.053826,-0.117873,-0.064030,-0.100802,-0.023712,0.002013


In [238]:
# Creation des figures avec Plotly (en suivant les indications dans le template du blog et les tutos sur le site officiel)
df=new_df
rinkpath = "/content/LNH-data-science/data_science_project/figures/nhl_rink.png"

# create figure
fig = go.Figure()

# Widget pour choisir la saison
season_widget = Dropdown(
    options=sorted(df['season'].unique()),
    description='Season: ',
    disabled=False,
)

# Widget pour choisir l'equipe
team_widget = Dropdown(
    options=sorted(df['team'].unique()),
    description='Team: ',
    disabled=False,
)

RINK_EXTENT = [-100, 100, -42.5, 42.5]
# Add images
fig.add_layout_image(
        dict(
            source=str(rink_path),
            xref="paper",
            yref="paper",
            x=0,
            y=0,
            sizex=2,
            sizey=2,
            sizing="stretch",
            opacity=1,
            layer="below")
)

# Update plot sizing
fig.update_layout(
    width=800,
    height=900,
    autosize=False,
    margin=dict(t=0, b=0, l=0, r=0),
    template="plotly_white",
)

# Update 3D scene options
fig.update_scenes(
    aspectratio=dict(x=1, y=1, z=0.7),
    aspectmode="manual"
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Trace type:", showarrow=False,
        x=0, y=1.085, yref="paper", align="left")
    ]
)

@interact(season=season_widget, team=team_widget)
def update_plot(season, team):
    # Pour une equipe et une saison donnee, pour afficher nos donnees, on peut les mettre dans une matrice de niveau (qui represente une sorte de heatmap), il faut donc pivoter la table
    z = (
        new_df.query('season == @season and team == @team').pivot_table(index="y", columns="x", values="shot_per_team-hr", aggfunc='sum')
          .reindex(index=range(10), columns=range(20))
          .to_numpy()
    )

    # Il faut corriger ca
    x = np.arange(20)
    y = np.arange(10)

    # Add surface trace
    fig = go.Figure(go.Contour(
        z=z,
        x=x,
        y=y,
        ncontours=16,
        contours=dict(coloring="fill", showlines=True),
        colorscale="Turbo",
        opacity=0.65,
    ))
    #fig.add_trace(go.Contour(z=df.values.tolist(), colorscale="Viridis"))


    return fig

interactive(children=(Dropdown(description='Season: ', options=('2016', '2017', '2018', '2019', '2020', '2021'…

In [240]:
# Je comprends pas bien comment les dropdowns dans Plotly fonctionnent,
# contrairement a iPyWidgets et le decorateur interact

# create figure
fig = go.Figure()

# Add surface trace
fig.add_trace(go.Surface(z=z.tolist(), colorscale="Viridis"))

# Update plot sizing
fig.update_layout(
    width=800,
    height=900,
    autosize=False,
    margin=dict(t=0, b=0, l=0, r=0),
    template="plotly_white",
)

# Update 3D scene options
fig.update_scenes(
    aspectratio=dict(x=1, y=1, z=0.7),
    aspectmode="manual"
)

# Add dropdown
fig.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=["type", "surface"],
                    label="3D Surface",
                    method="restyle"
                ),
                dict(
                    args=["type", "heatmap"],
                    label="Heatmap",
                    method="restyle"
                )
            ]),
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.1,
            yanchor="top"
        ),
    ]
)

# Add annotation
fig.update_layout(
    annotations=[
        dict(text="Trace type:", showarrow=False,
        x=0, y=1.085, yref="paper", align="left")
    ]
)

fig.show()